In [5]:
sample_text = """Artificial Intelligence is rapidly evolving.
    
It is transforming industries such as healthcare, finance, and education. Machine learning, a subset of AI, involves training models on vast amounts of data.
    
Natural Language Processing allows computers to understand human language. This has led to the rise of advanced chatbots and virtual assistants."""

# Fixed character text splitter

In [6]:
from langchain_text_splitters import CharacterTextSplitter

def fixed_character_split(text):
    print("--- Fixed Character Text Splitter ---")
    
    # separator: The character to split on
    # chunk_size: Maximum number of characters per chunk
    # chunk_overlap: Number of characters to overlap between chunks to maintain context
    text_splitter = CharacterTextSplitter(
        separator="\n\n",
        chunk_size=150,
        chunk_overlap=20,
        length_function=len,
        is_separator_regex=False
    )
    
    chunks = text_splitter.split_text(text)
    
    for i, chunk in enumerate(chunks):
        print(f"Chunk {i+1} (Length: {len(chunk)}): {chunk}\n")
    return chunks

fixed_character_split(sample_text)

Created a chunk of size 157, which is longer than the specified 150


--- Fixed Character Text Splitter ---
Chunk 1 (Length: 44): Artificial Intelligence is rapidly evolving.

Chunk 2 (Length: 157): It is transforming industries such as healthcare, finance, and education. Machine learning, a subset of AI, involves training models on vast amounts of data.

Chunk 3 (Length: 144): Natural Language Processing allows computers to understand human language. This has led to the rise of advanced chatbots and virtual assistants.



['Artificial Intelligence is rapidly evolving.',
 'It is transforming industries such as healthcare, finance, and education. Machine learning, a subset of AI, involves training models on vast amounts of data.',
 'Natural Language Processing allows computers to understand human language. This has led to the rise of advanced chatbots and virtual assistants.']

# Recursive character text splitter

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def recursive_character_split(text):
    print("--- Recursive Character Text Splitter ---")
    
    text_splitter = RecursiveCharacterTextSplitter(
        # These are the default separators. It tries \n\n first, then \n, then spaces.
        separators=["\n\n", "\n", " ", ""],
        chunk_size=150,
        chunk_overlap=20,
        length_function=len
    )
    
    chunks = text_splitter.split_text(text)
    
    for i, chunk in enumerate(chunks):
        print(f"Chunk {i+1} (Length: {len(chunk)}): {chunk}\n")
    return chunks

recursive_character_split(sample_text)

# Token based text splitter

In [ ]:
from langchain_text_splitters import TokenTextSplitter

def token_based_split(text):
    print("--- Token Based Text Splitter ---")
    
    # chunk_size and chunk_overlap are now measured in TOKENS, not characters
    text_splitter = TokenTextSplitter(
        encoding_name="cl100k_base", # The encoding used by GPT-4 and GPT-3.5
        chunk_size=30, 
        chunk_overlap=5
    )
    
    TokenTextSplitter.from_tiktoken_encoder(model_name='gpt-4o')
    
    chunks = text_splitter.split_text(text)
    
    for i, chunk in enumerate(chunks):
        print(f"Chunk {i+1}: {chunk}\n")
    return chunks

token_based_split(sample_text)

# File based splitter

In [1]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

def file_based_markdown_split(markdown_text):
    print("--- File-Based (Markdown Header) Text Splitter ---")
    
    # Define which headers we want to split on and what to name them in metadata
    headers_to_split_on = [
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3"),
    ]
    
    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers_to_split_on,
        strip_headers=False # Set to True if you want to remove the markdown "#" symbols from the output
    )
    
    # This returns LangChain Document objects, not just raw strings
    documents = markdown_splitter.split_text(markdown_text)
    
    for i, doc in enumerate(documents):
        print(f"Chunk {i+1} Metadata: {doc.metadata}")
        print(f"Content: {doc.page_content}\n")
    return documents

text = """# Introduction to AI
Artificial Intelligence is rapidly evolving and changing the world.
    
## Machine Learning
Machine learning involves training models on vast amounts of data to recognize patterns.
    
## Deep Learning
Deep learning uses neural networks with many layers to solve complex problems."""

file_based_markdown_split(text)

d:\IT\AI\teaching\GenAI_course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- File-Based (Markdown Header) Text Splitter ---
Chunk 1 Metadata: {'Header 1': 'Introduction to AI'}
Content: # Introduction to AI
Artificial Intelligence is rapidly evolving and changing the world.

Chunk 2 Metadata: {'Header 1': 'Introduction to AI', 'Header 2': 'Machine Learning'}
Content: ## Machine Learning
Machine learning involves training models on vast amounts of data to recognize patterns.

Chunk 3 Metadata: {'Header 1': 'Introduction to AI', 'Header 2': 'Deep Learning'}
Content: ## Deep Learning
Deep learning uses neural networks with many layers to solve complex problems.



[Document(metadata={'Header 1': 'Introduction to AI'}, page_content='# Introduction to AI\nArtificial Intelligence is rapidly evolving and changing the world.'),
 Document(metadata={'Header 1': 'Introduction to AI', 'Header 2': 'Machine Learning'}, page_content='## Machine Learning\nMachine learning involves training models on vast amounts of data to recognize patterns.'),
 Document(metadata={'Header 1': 'Introduction to AI', 'Header 2': 'Deep Learning'}, page_content='## Deep Learning\nDeep learning uses neural networks with many layers to solve complex problems.')]

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language

def split_python_code(input):
    # Create the Python-aware text splitter
    # We set a small chunk_size (150) to force the splitter to break the code into multiple chunks.
    # It will prioritize splitting at class and function definitions.
    python_splitter = RecursiveCharacterTextSplitter.from_language(
        language=Language.PYTHON,
        chunk_size=500,
        chunk_overlap=0 
    )

    # 3. Split the code
    code_chunks = python_splitter.split_text(input)

    # 4. Display the results
    print(f"Total chunks created: {len(code_chunks)}\n")
    for i, chunk in enumerate(code_chunks):
        print(f"--- Chunk {i+1} (Length: {len(chunk)}) ---")
        print(chunk.strip())
        print("-" * 40 + "\n")

python_code = """
# math_operations.py
# This is a sample Python file used to demonstrate the text splitter.

class Calculator:
    def __init__(self):
        self.history = []

    def add(self, a, b):
        \"\"\"Adds two numbers and stores the history.\"\"\"
        result = a + b
        self.history.append(f"Added {a} + {b} = {result}")
        return result

    def subtract(self, a, b):
        \"\"\"Subtracts two numbers and stores the history.\"\"\"
        result = a - b
        self.history.append(f"Subtract {a} - {b} = {result}")
        return result

def print_welcome_message():
    print("Welcome to the Calculator Module!")
    print("You can use this to perform basic arithmetic.")
    """

split_python_code(python_code)

Total chunks created: 3

--- Chunk 1 (Length: 90) ---
# math_operations.py
# This is a sample Python file used to demonstrate the text splitter.
----------------------------------------

--- Chunk 2 (Length: 450) ---
class Calculator:
    def __init__(self):
        self.history = []

    def add(self, a, b):
        """Adds two numbers and stores the history."""
        result = a + b
        self.history.append(f"Added {a} + {b} = {result}")
        return result

    def subtract(self, a, b):
        """Subtracts two numbers and stores the history."""
        result = a - b
        self.history.append(f"Subtract {a} - {b} = {result}")
        return result
----------------------------------------

--- Chunk 3 (Length: 134) ---
def print_welcome_message():
    print("Welcome to the Calculator Module!")
    print("You can use this to perform basic arithmetic.")
----------------------------------------



# Semantic chunking

In [3]:
import os
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from dotenv import load_dotenv

load_dotenv(r'D:\IT\AI\teaching\GenAI_course\.env')

def semantic_chunking_example():   
        
    embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
    
    # Initialize the Semantic Chunker
    # breakpoint_threshold_type determines how the splitter decides to break.
    # Options: "percentile" (default), "standard_deviation", "interquartile", or "gradient"
    text_splitter = SemanticChunker(
        embeddings,
        breakpoint_threshold_type="percentile",
        breakpoint_threshold_amount=98 # Adjust this to make splits more or less frequent
    )
    
    # Create a sample text with clear topical shifts
    sample_text = (
        "The Apollo 11 mission was the first manned mission to land on the Moon. "
        "Neil Armstrong and Buzz Aldrin walked on the lunar surface in 1969. "
        "It was a monumental achievement for human space exploration. "
        "In completely unrelated news, baking a perfect chocolate chip cookie requires precision. "
        "You need to cream the butter and sugar together until light and fluffy. "
        "Adding a dash of sea salt on top enhances the chocolate flavor. "
        "Furthermore, artificial intelligence is reshaping the modern economy. "
        "Machine learning models can now generate text, images, and even code. "
        "This technological leap is being compared to the industrial revolution."
    )
    
    # Split the text
    # The chunker will group the space sentences, the baking sentences, and the AI sentences.
    docs = text_splitter.create_documents([sample_text])
    
    # 5. Display the results
    print(f"Total chunks created: {len(docs)}\n")
    for i, doc in enumerate(docs):
        print(f"--- Chunk {i+1} ---")
        print(doc.page_content)
        print("-" * 20 + "\n")

if __name__ == "__main__":
    semantic_chunking_example()

Total chunks created: 2

--- Chunk 1 ---
The Apollo 11 mission was the first manned mission to land on the Moon. Neil Armstrong and Buzz Aldrin walked on the lunar surface in 1969. It was a monumental achievement for human space exploration. In completely unrelated news, baking a perfect chocolate chip cookie requires precision. You need to cream the butter and sugar together until light and fluffy. Adding a dash of sea salt on top enhances the chocolate flavor. Furthermore, artificial intelligence is reshaping the modern economy.
--------------------

--- Chunk 2 ---
Machine learning models can now generate text, images, and even code. This technological leap is being compared to the industrial revolution.
--------------------

